# Module 2.8 — Data Serialisation & File Formats

### Python Mastery · Track 2: Data Structures & Iteration

Serialisation means converting data structures into a form that can be stored or transmitted, and reading them back. Different formats suit different needs. This module covers JSON, CSV, pickle, configuration files, and a first look at XML, with their trade-offs and safety considerations.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it. The examples write small files in the working folder so every cell is reproducible.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Convert data to and from JSON, including custom types.
2. Read and write CSV data, including with headers.
3. Use pickle for Python objects and understand its security risk.
4. Read configuration with `configparser` and TOML with `tomllib`.
5. Produce and parse simple XML.

**Prerequisites:** Track 1 (especially Module 1.8) and Modules 2.1 to 2.7.

---

## Part 1 · JSON — The Common Interchange Format

JSON (JavaScript Object Notation) is the most widely used format for exchanging data between programs and across the web. It maps cleanly onto Python types: objects become dicts, arrays become lists, and strings, numbers, booleans, and null map to their Python equivalents.

The `json` module offers four functions: `dumps` (to a string), `dump` (to a file), `loads` (from a string), and `load` (from a file). The `s` stands for "string."

In [ ]:
import json

data = {
    "name": "Asha",
    "age": 30,
    "skills": ["python", "sql"],
    "active": True,
    "manager": None,
}

# To a string. indent makes it human-readable; sort_keys orders the keys.
text = json.dumps(data, indent=2, sort_keys=True)
print(text)

# Back from a string into Python objects.
restored = json.loads(text)
print("restored type:", type(restored), "| skills:", restored["skills"])

In [ ]:
import json

# Writing to and reading from a file.
with open("person.json", "w") as f:
    json.dump(data, f, indent=2)

with open("person.json", "r") as f:
    loaded = json.load(f)
print("loaded from file:", loaded["name"], loaded["age"])

JSON understands only its basic types. To serialise something like a `datetime`, supply a `default` function that converts unknown objects into something JSON can represent, and reverse the conversion when loading.

In [ ]:
import json
from datetime import datetime

record = {"event": "login", "at": datetime(2025, 5, 30, 9, 15)}

def encode_unknown(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()          # represent a datetime as an ISO string
    raise TypeError(f"cannot serialise {type(obj)}")

text = json.dumps(record, default=encode_unknown)
print("encoded:", text)

# On load, convert the known field back to a datetime.
loaded = json.loads(text)
loaded["at"] = datetime.fromisoformat(loaded["at"])
print("decoded 'at' is a datetime:", loaded["at"], type(loaded["at"]))

## Part 2 · CSV — Tabular Data

CSV (comma-separated values) stores rows of columns and is the lingua franca of spreadsheets. The `csv` module handles the details of quoting and separators that trip up naive string splitting. Use `csv.writer`/`csv.reader` for plain rows, and `csv.DictWriter`/`csv.DictReader` to work with named columns, which is clearer.

When writing CSV files, open them with `newline=""` to avoid blank lines on some platforms.

In [ ]:
import csv

rows = [
    {"name": "Asha", "city": "Pune", "score": 88},
    {"name": "Ben", "city": "Delhi", "score": 92},
    {"name": "Chen", "city": "Pune", "score": 79},
]

# Writing with named columns.
with open("people.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "city", "score"])
    writer.writeheader()                 # write the column titles
    writer.writerows(rows)

# Show the raw file contents.
with open("people.csv") as f:
    print(f.read())

In [ ]:
import csv

# Reading back with DictReader: each row becomes a dict keyed by the header.
with open("people.csv", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Note: every value read from CSV is text; convert numbers explicitly.
        print(row["name"], "in", row["city"], "scored", int(row["score"]))

## Part 3 · pickle — Python Objects, with a Serious Caveat

`pickle` serialises almost any Python object, including custom classes, into bytes and back. It is convenient for saving program state between runs.

However, **unpickling data executes code embedded in it**, so loading a pickle from an untrusted source can run arbitrary, possibly malicious, instructions. The rule is simple: never unpickle data you did not create or do not fully trust. For data shared with others or across languages, prefer JSON.

In [ ]:
import pickle

state = {"level": 5, "inventory": ["sword", "shield"], "scores": (10, 20, 30)}

# Serialise to bytes and back. Note that pickle preserves the tuple as a tuple,
# whereas JSON would turn it into a list.
blob = pickle.dumps(state)
print("pickled bytes (first 30):", blob[:30])

restored = pickle.loads(blob)
print("restored:", restored)
print("tuple preserved:", type(restored["scores"]))

# Saving to a file uses binary mode.
with open("state.pkl", "wb") as f:
    pickle.dump(state, f)
with open("state.pkl", "rb") as f:
    print("from file:", pickle.load(f))

## Part 4 · Configuration: `configparser` and TOML

Programs often read settings from a configuration file. `configparser` handles the classic INI style, with sections and key-value pairs. TOML is a modern, clearer alternative; Python 3.11+ can read it with the built-in `tomllib` (read-only). Both keep configuration separate from code, which is good practice.

In [ ]:
import configparser
from io import StringIO

ini_text = """
[database]
host = localhost
port = 5432

[features]
debug = true
retries = 3
"""

config = configparser.ConfigParser()
config.read_string(ini_text)            # read from text (read_file/read for files)

print("host:", config["database"]["host"])
print("port as int:", config.getint("database", "port"))
print("debug as bool:", config.getboolean("features", "debug"))
print("sections:", config.sections())

In [ ]:
import tomllib       # standard library, read-only, Python 3.11+

toml_text = """
title = "My App"

[server]
host = "0.0.0.0"
port = 8000
enabled = true

[server.limits]
max_connections = 100
"""

# tomllib.loads reads a string; tomllib.load reads a binary file opened with "rb".
settings = tomllib.loads(toml_text)
print("title:", settings["title"])
print("server port:", settings["server"]["port"])
print("nested limit:", settings["server"]["limits"]["max_connections"])

## Part 5 · XML — A First Look

XML is a verbose but expressive format still common in documents and older systems. The standard `xml.etree.ElementTree` module builds and reads XML as a tree of elements, each with a tag, optional attributes, text, and children.

A safety note: when parsing XML from untrusted sources, be aware that XML can carry attacks (such as entity-expansion attacks); for hostile input, a hardened parser library is advisable.

In [ ]:
import xml.etree.ElementTree as ET

# Build a small document as a tree.
catalog = ET.Element("catalog")
for title, price in [("Python Basics", "29.99"), ("Advanced Python", "49.99")]:
    book = ET.SubElement(catalog, "book")
    ET.SubElement(book, "title").text = title
    ET.SubElement(book, "price").text = price

xml_bytes = ET.tostring(catalog, encoding="unicode")
print(xml_bytes)

In [ ]:
import xml.etree.ElementTree as ET

xml_data = """
<catalog>
  <book><title>Python Basics</title><price>29.99</price></book>
  <book><title>Advanced Python</title><price>49.99</price></book>
</catalog>
"""

root = ET.fromstring(xml_data)          # parse a string into an element tree
for book in root.findall("book"):       # iterate over child elements named 'book'
    title = book.find("title").text
    price = float(book.find("price").text)
    print(f"{title}: {price}")

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Round-tripping JSON (Easy)

In [ ]:
import json
data = {"city": "Pune", "population": 7000000, "coastal": False}
text = json.dumps(data)
print("as text:", text)
print("back to dict:", json.loads(text))
# Experiment: add indent=2 to dumps for a readable layout.

### Example 2 — Writing and reading a CSV (Easy)

In [ ]:
import csv

with open("scores.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "score"])      # header
    writer.writerow(["Asha", 88])
    writer.writerow(["Ben", 92])

with open("scores.csv", newline="") as f:
    for row in csv.reader(f):
        print(row)

### Example 3 — JSON with a custom type (Medium)

In [ ]:
import json
from datetime import date

events = [
    {"name": "launch", "on": date(2025, 1, 10)},
    {"name": "review", "on": date(2025, 3, 5)},
]

def default(obj):
    if isinstance(obj, date):
        return obj.isoformat()
    raise TypeError("unserialisable")

text = json.dumps(events, default=default, indent=2)
print(text)

### Example 4 — Summarising a CSV with DictReader (Medium)

In [ ]:
import csv
from collections import defaultdict

# Reuse people.csv written in Part 2 (name, city, score).
totals = defaultdict(list)
with open("people.csv", newline="") as f:
    for row in csv.DictReader(f):
        totals[row["city"]].append(int(row["score"]))

for city, scores in totals.items():
    print(f"{city}: average {sum(scores) / len(scores):.1f} over {len(scores)} people")

### Example 5 — Saving and restoring program state with pickle (Difficult)

In [ ]:
import pickle

class Game:
    def __init__(self, level, score):
        self.level = level
        self.score = score
    def __repr__(self):
        return f"Game(level={self.level}, score={self.score})"

original = Game(level=7, score=2500)

# Persist the whole object to a file, then restore it in a later run.
with open("game.pkl", "wb") as f:
    pickle.dump(original, f)

with open("game.pkl", "rb") as f:
    loaded = pickle.load(f)

print("restored object:", loaded, "| same data:", loaded.level == original.level)
# Reminder: only ever unpickle files you trust.

### Example 6 — Reading layered configuration (Difficult)

In [ ]:
import tomllib

toml_text = """
[app]
name = "Reporter"
version = "2.1"

[app.logging]
level = "INFO"
to_file = true

[[app.outputs]]
type = "pdf"

[[app.outputs]]
type = "html"
"""

config = tomllib.loads(toml_text)
app = config["app"]
print("name:", app["name"], "version:", app["version"])
print("log level:", app["logging"]["level"])
print("output types:", [o["type"] for o in app["outputs"]])   # a list of tables

---

## Exercises

**Exercise 1 (Easy).** Convert the dictionary below to a nicely indented JSON string and print it.

In [ ]:
import json
book = {"title": "Dune", "author": "Herbert", "year": 1965, "tags": ["sci-fi", "classic"]}
# Your solution here


**Exercise 2 (Easy).** Write the three rows below to a CSV file named `cities.csv` with a header row, then read it back and print each row.

In [ ]:
import csv
cities = [["Pune", 7000000], ["Delhi", 19000000], ["Tokyo", 37000000]]
# Your solution here (header: city, population)


**Exercise 3 (Medium).** Given the JSON string below, parse it and print the name of each user along with how many roles they have.

In [ ]:
import json
payload = '{"users": [{"name": "Asha", "roles": ["admin", "editor"]}, {"name": "Ben", "roles": ["viewer"]}]}'
# Your solution here


**Exercise 4 (Medium).** Read the file `people.csv` (written earlier in this notebook) with `DictReader` and print only the names of people whose score is 85 or above.

In [ ]:
import csv
# Your solution here


**Exercise 5 (Difficult).** Parse the XML below and build a dictionary mapping each product name to its price as a float.

In [ ]:
import xml.etree.ElementTree as ET
xml_data = """
<store>
  <product><name>Pen</name><price>15.00</price></product>
  <product><name>Notebook</name><price>45.50</price></product>
  <product><name>Eraser</name><price>5.25</price></product>
</store>
"""
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import json
book = {"title": "Dune", "author": "Herbert", "year": 1965, "tags": ["sci-fi", "classic"]}
print(json.dumps(book, indent=2))

**Solution 2**

In [ ]:
import csv
cities = [["Pune", 7000000], ["Delhi", 19000000], ["Tokyo", 37000000]]
with open("cities.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["city", "population"])
    writer.writerows(cities)

with open("cities.csv", newline="") as f:
    for row in csv.reader(f):
        print(row)

**Solution 3**

In [ ]:
import json
payload = '{"users": [{"name": "Asha", "roles": ["admin", "editor"]}, {"name": "Ben", "roles": ["viewer"]}]}'
data = json.loads(payload)
for user in data["users"]:
    print(user["name"], "has", len(user["roles"]), "roles")

**Solution 4**

In [ ]:
import csv
with open("people.csv", newline="") as f:
    for row in csv.DictReader(f):
        if int(row["score"]) >= 85:
            print(row["name"])

**Solution 5**

In [ ]:
import xml.etree.ElementTree as ET
xml_data = """
<store>
  <product><name>Pen</name><price>15.00</price></product>
  <product><name>Notebook</name><price>45.50</price></product>
  <product><name>Eraser</name><price>5.25</price></product>
</store>
"""
root = ET.fromstring(xml_data)
prices = {p.find("name").text: float(p.find("price").text) for p in root.findall("product")}
print(prices)

---

## Summary and Key Points

- JSON is the common interchange format; use `dumps`/`loads` for strings and `dump`/`load` for files, and a `default` function to serialise types such as `datetime`.
- CSV stores tabular data; `DictWriter` and `DictReader` work with named columns, every value reads back as text, and files should be opened with `newline=""`.
- pickle serialises almost any Python object and preserves types such as tuples, but unpickling untrusted data can execute arbitrary code, so only load pickles you trust.
- Configuration belongs outside code: `configparser` reads INI files, and `tomllib` reads TOML (read-only) in Python 3.11+.
- `xml.etree.ElementTree` builds and parses XML as a tree; treat untrusted XML with caution.

### Track 2 complete

This concludes Track 2, Data Structures & Iteration. You can now choose and use the core containers, build them concisely with comprehensions, write iterators and generators, apply specialised collections, sort and search efficiently, process text with regular expressions, handle dates and times correctly, and serialise data in the major formats. Track 3 builds on this with object-oriented programming and design patterns.